In [ ]:
import os, sys, subprocess, shlex, pathlib

SCRIPT = "TrainRFDETR_SOLO_SemiSupervised_STATIC_Ucloud.py"

def _detect_runtime_profile() -> str:
    forced = os.environ.get("RFDETR_RUNTIME_PROFILE", "auto").strip().lower()
    if forced in {"ucloud", "local"}:
        return forced
    if os.name != "nt" and pathlib.Path("/work").exists():
        has_member = any(pathlib.Path("/work").glob("Member Files:*"))
        has_hash = any(pathlib.Path("/work").glob("*#*"))
        if has_member or has_hash:
            return "ucloud"
    return "local"

RUNTIME_PROFILE = _detect_runtime_profile()
os.environ["RFDETR_RUNTIME_PROFILE"] = RUNTIME_PROFILE
if RUNTIME_PROFILE == "ucloud":
    PROJECT_DIR = os.environ.get("PROJECT_DIR", "/work/projects/myproj/SOLO_Supervised_RFDETR/")
else:
    PROJECT_DIR = os.environ.get("PROJECT_DIR", str(pathlib.Path.cwd() / "SOLO_Supervised_RFDETR"))

PROJECT_PATH = pathlib.Path(PROJECT_DIR).resolve()
REPO_ROOT = PROJECT_PATH.parent
DATASET_ROOT = pathlib.Path(os.environ.get("STAT_DATASETS_ROOT", str(PROJECT_PATH / "Stat_Dataset"))).resolve()

def _latest_dataset_dir(root: pathlib.Path, token: str) -> str:
    cands = sorted([p for p in root.glob(f"*{token}*") if p.is_dir()])
    return str(cands[-1]) if cands else ""

os.environ["DATASET_EPI"] = os.environ.get("DATASET_EPI", "").strip() or _latest_dataset_dir(DATASET_ROOT, "SquamousEpithelialCell_OVR")
os.environ["DATASET_LEUCO"] = os.environ.get("DATASET_LEUCO", "").strip() or _latest_dataset_dir(DATASET_ROOT, "Leucocyte_OVR")

if RUNTIME_PROFILE == "ucloud":
    user_base = os.environ.get("USER_BASE_DIR", "").strip()
    if not user_base:
        work = pathlib.Path("/work")
        members = sorted(work.glob("Member Files:*"))
        hashed = sorted([p for p in work.glob("*#*") if p.is_dir()])
        user_base = members[0].name if members else (hashed[0].name if hashed else "")
    if user_base:
        os.environ.setdefault("IMAGES_FALLBACK_ROOT", f"/work/{user_base}/CellScanData/Zoom10x - Quality Assessment_Cleaned")
        os.environ.setdefault("OUTPUT_ROOT", f"/work/{user_base}/RFDETR_SOLO_OUTPUT/HPO_BOTH_OVR")
else:
    os.environ.setdefault("IMAGES_FALLBACK_ROOT", str(pathlib.Path(r"D:\\PHD\\PhdData\\CellScanData\\Zoom10x - Quality Assessment_Cleaned")))
    os.environ.setdefault("OUTPUT_ROOT", str(PROJECT_PATH / "RFDETR_SOLO_OUTPUT" / "HPO_BOTH_OVR_local"))

# Main semi-supervised experiment knobs
os.environ["RFDETR_SS_TARGET"] = "epi"
os.environ["RFDETR_SS_INIT_MODE"] = "scratch"
os.environ["RFDETR_SS_TRAIN_FRACTIONS"] = "0.25,0.10,0.05"
os.environ["RFDETR_SS_SEEDS"] = "42,43,44"
os.environ["RFDETR_SS_FRACTION_SEEDS"] = "42"
os.environ.setdefault("RFDETR_SS_PSEUDO_SCORE_THRESH", "0.70")
os.environ.setdefault("RFDETR_SS_PSEUDO_IMAGES_PER_LABELED", "6")
os.environ.setdefault("RFDETR_SS_PSEUDO_MAX_DETS_PER_IMAGE", "50")
os.environ.setdefault("RFDETR_SS_UNLABELED_MAX_IMAGES", "0")

# Keep supervised hyperparameters aligned with your current static RFDETR setup
os.environ.setdefault("RFDETR_INPUT_MODE", "640")
os.environ.setdefault("RFDETR_USE_PATCH_224", "0")
os.environ.setdefault("RFDETR_MODEL_CLS", "RFDETRLarge")
os.environ.setdefault("RFDETR_EPI_EPOCHS", "50")
os.environ.setdefault("RFDETR_EPI_LR", "5e-5")
os.environ.setdefault("RFDETR_GRAD_ACCUM_STEPS", "1")
os.environ.setdefault("RFDETR_WEIGHT_DECAY", "7e-4")
os.environ.setdefault("RFDETR_DROPOUT", "0.15")
os.environ.setdefault("RFDETR_EPI_BATCH", "4")
os.environ.setdefault("RFDETR_EPI_NUM_QUERIES", "120")
os.environ.setdefault("RFDETR_AUG_COPIES", "0")
os.environ.setdefault("RFDETR_EARLY_STOPPING", "1")
os.environ.setdefault("RFDETR_EARLY_STOPPING_PATIENCE", "10")
os.environ.setdefault("RFDETR_EARLY_STOPPING_MIN_DELTA", "0.001")
os.environ.setdefault("RFDETR_EARLY_STOPPING_USE_EMA", "0")

for k in ["DATASET_EPI", "DATASET_LEUCO", "IMAGES_FALLBACK_ROOT", "OUTPUT_ROOT"]:
    p = os.environ.get(k, "")
    if not p:
        raise ValueError(f"{k} is empty")
    if not pathlib.Path(p).exists():
        raise FileNotFoundError(f"Configured path does not exist for {k}: {p}")

print("RFDETR_RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("PROJECT_DIR =", PROJECT_DIR)
for k in [
    "DATASET_EPI", "DATASET_LEUCO", "IMAGES_FALLBACK_ROOT", "OUTPUT_ROOT",
    "RFDETR_SS_TARGET", "RFDETR_SS_INIT_MODE", "RFDETR_SS_TRAIN_FRACTIONS",
    "RFDETR_SS_SEEDS", "RFDETR_SS_FRACTION_SEEDS",
    "RFDETR_SS_PSEUDO_SCORE_THRESH", "RFDETR_SS_PSEUDO_IMAGES_PER_LABELED",
    "RFDETR_SS_PSEUDO_MAX_DETS_PER_IMAGE", "RFDETR_SS_UNLABELED_MAX_IMAGES",
    "RFDETR_INPUT_MODE", "RFDETR_MODEL_CLS"
]:
    print(f"{k} = {os.environ.get(k)}")

wd = pathlib.Path(PROJECT_DIR).resolve()
script_path = wd / SCRIPT
if not script_path.exists():
    raise FileNotFoundError(f"Missing script: {script_path}")

cmd = [sys.executable, "-u", str(script_path)]
print("[LAUNCH]")
print(" cwd:", wd)
print(" cmd:", " ".join(shlex.quote(x) for x in cmd))

proc = subprocess.Popen(cmd, cwd=str(wd), env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Run failed with exit code {ret}")
